# Econ 390 - Lecture 18: Groupby and Aggregation
Today we will be going over how to aggregate data and apply the Split-Apply-Combine method using Python. All of [Chapter 10 of McKinney](wesmckinney.com/book/data-aggregation) and much of [Data Transformation of Turrell](https://aeturrell.github.io/coding-for-economists/data-transformation.html) are applicable here, though we will not be going over all of what they cover.

In [1]:
import pandas as pd

url = "https://m-mcmain.github.io/files/Econ390SP26/"

## Split-Apply-Combine
The Split-Apply-Combine method is so common there are different images describing how to use it in both of the textbooks:

![McKinney SAC](https://wesmckinney.com/book/images/pda3_1001.png)
![Turrell SAC](https://jakevdp.github.io/PythonDataScienceHandbook/figures/03.08-split-apply-combine.png)

In [3]:
# NLSY variables
variables = {"R0000100":"ID", "R0536300":"Sex", "U4950200":"Education", 
             "U5753500":"Income", "Z9061912":"Weeks Worked"}
# "R0536401":"Birth Month", "R0536402":"Birth Year", "R1482600":"Race", "U4943000":"Age"

In [4]:
# Read in data
nlsy = pd.read_csv(url + "nlsy_wage_educSex.csv", na_values=-5, usecols=variables.keys()).rename(columns=variables)
nlsy

,ID,Sex,Education,Income,Weeks Worked
0,1,2,NaN,NaN,-4
1,2,1,2.0,115000.0,46
2,3,2,4.0,-4.0,0
3,4,2,2.0,45000.0,51
4,5,1,2.0,150000.0,43
...,...,...,...,...,...
8979,9018,2,1.0,90000.0,52
8980,9019,1,3.0,43000.0,43
8981,9020,1,NaN,NaN,-4
8982,9021,1,4.0,47000.0,52


In [5]:
# Info
nlsy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8984 entries, 0 to 8983
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            8984 non-null   int64  
 1   Sex           8984 non-null   int64  
 2   Education     6713 non-null   float64
 3   Income        6713 non-null   float64
 4   Weeks Worked  8984 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 351.1 KB


In [9]:
# Counts of different groups
print(nlsy["Sex"].value_counts())
print(nlsy["Education"].value_counts())

Sex
1    4599
2    4385
Name: count, dtype: int64
Education
 2.0    2563
 4.0    1351
 1.0     871
 3.0     621
 5.0     614
 0.0     502
 7.0     106
 6.0      59
-3.0      26
Name: count, dtype: int64


In [10]:
# Translating codes to what they mean
educ_codes = {0:"None", 1:"GED", 2:"High School", 3:"Associate's", 4:"Bachelor's", 5:"Master's",
              6:"PhD", 7:"Professional"}
sex_codes = {1:"Male", 2:"Female", 0:"No Information"}

nlsy["Education"] = nlsy["Education"].replace(educ_codes)
nlsy["Sex"] = nlsy["Sex"].replace(sex_codes)

nlsy

,ID,Sex,Education,Income,Weeks Worked
0,1,Female,NaN,NaN,-4
1,2,Male,High School,115000.0,46
2,3,Female,Bachelor's,-4.0,0
3,4,Female,High School,45000.0,51
4,5,Male,High School,150000.0,43
...,...,...,...,...,...
8979,9018,Female,GED,90000.0,52
8980,9019,Male,Associate's,43000.0,43
8981,9020,Male,NaN,NaN,-4
8982,9021,Male,Bachelor's,47000.0,52


In [11]:
# Set the index and groupby
nlsy.set_index("ID", inplace=True)
nlsy_grouped = nlsy.groupby("Sex")

In [12]:
# Get Group method
nlsy_grouped.get_group("Female")

,Sex,Education,Income,Weeks Worked
ID,,,,
1,Female,NaN,NaN,-4
3,Female,Bachelor's,-4.0,0
4,Female,High School,45000.0,51
6,Female,High School,25000.0,40
8,Female,Master's,70000.0,52
...,...,...,...,...
9014,Female,None,-4.0,22
9015,Female,Master's,96000.0,34
9016,Female,Bachelor's,60000.0,48


In [13]:
# Mean
nlsy.mean(numeric_only=True)

Income          48722.649784
Weeks Worked       25.640583
dtype: float64

In [14]:
# Mean of grouped
nlsy_grouped.mean(numeric_only=True)

,Income,Weeks Worked
Sex,,
Female,39800.302482,25.797719
Male,58016.761557,25.490759


In [27]:
# Limit to just people working
nlsy_working = nlsy.loc[nlsy["Weeks Worked"] > 0]
nlsy_working

,Sex,Education,Income,Weeks Worked
ID,,,,
2,Male,High School,115000.0,46
4,Female,High School,45000.0,51
5,Male,High School,150000.0,43
6,Female,High School,25000.0,40
8,Female,Master's,70000.0,52
...,...,...,...,...
9016,Female,Bachelor's,60000.0,48
9018,Female,GED,90000.0,52
9019,Male,Associate's,43000.0,43


In [17]:
# Let's make it grouped!
nlsy_meanSex_grouped = nlsy.loc[nlsy["Weeks Worked"] > 0].groupby("Sex").mean(numeric_only = True)
nlsy_meanSex_grouped

,Income,Weeks Worked
Sex,,
Female,48450.467860,42.751278
Male,65821.912625,43.738944


In [28]:
# Cleaning and new Value
nlsy_working = nlsy_working.loc[nlsy_working["Education"] != -3.0]
nlsy_working.loc[(nlsy_working["Education"] == "High School") | (nlsy_working["Education"] == "None") | (nlsy_working["Education"] == "GED"),"Education"] = "High School and Below"
nlsy_working.loc[(nlsy_working["Education"] == "Bachelor's") | (nlsy_working["Education"] == "Associate's"), "Education"] = "Bachelor's and Associate's"
nlsy_working.loc[(nlsy_working["Education"] != "High School and Below") & (nlsy_working["Education"] != "Bachelor's and Associate's"),"Education"] = "Master's and Up"
nlsy_working

,Sex,Education,Income,Weeks Worked
ID,,,,
2,Male,High School and Below,115000.0,46
4,Female,High School and Below,45000.0,51
5,Male,High School and Below,150000.0,43
6,Female,High School and Below,25000.0,40
8,Female,Master's and Up,70000.0,52
...,...,...,...,...
9016,Female,Bachelor's and Associate's,60000.0,48
9018,Female,High School and Below,90000.0,52
9019,Male,Bachelor's and Associate's,43000.0,43


In [29]:
print(nlsy_working["Education"].value_counts())

Education
High School and Below         3047
Bachelor's and Associate's    1744
Master's and Up                735
Name: count, dtype: int64


In [30]:
# Medians
nlsy_working.groupby("Education").median(numeric_only=True)

,Income,Weeks Worked
Education,,
Bachelor's and Associate's,55000.0,44.0
High School and Below,35000.0,44.0
Master's and Up,78000.0,45.0


Common aggregation methods are:
- `.mean()`
- `.sum()`
- `.std()`
- `.describe()`
- `.min()`
- `.max()`

## Practice: Split-Apply-Combine
Let's figure out how to get some measure of how variable annual income of workers are by education attainment.
1. Calculate the 75th percentile of Income for each new education type, storing the result in a DataFrame called q75 (So we split on what? And then apply what method?) *Hint: The .quantile() method computes percentiles.*
3. Calculate the 25th percentile for each type and store it in q25
4. Take the difference between these q75 and q25
5. Based on the IQR, which education type has the least amount of variation?

In [34]:
# Results:
nlsy_working_grouped = nlsy_working.groupby("Education")
q75 = nlsy_working_grouped["Income"].quantile(0.75)
q25 = nlsy_working_grouped["Income"].quantile(0.25)
q75 - q25

Education
Bachelor's and Associate's    54000.0
High School and Below         44000.0
Master's and Up               62000.0
Name: Income, dtype: float64

In [40]:
nlsy_working_grouped.describe()

Income                                            \
                             count          mean           std  min      25%   
Education                                                                      
Bachelor's and Associate's  1744.0  68138.235092  64234.270551 -4.0  32000.0   
High School and Below       3047.0  41395.575320  42969.892392 -4.0  13000.0   
Master's and Up              735.0  97691.197279  83286.984185 -4.0  53000.0   

                                                        Weeks Worked  \
                                50%       75%       max        count   
Education                                                              
Bachelor's and Associate's  55000.0   86000.0  380288.0       1744.0   
High School and Below       35000.0   57000.0  380288.0       3047.0   
Master's and Up             78000.0  115000.0  380288.0        735.0   

                                                                               
                                 mean        std  min   25%   50%   75%   max  
Education                                                                      
Bachelor's and Associate's  43.588876   8.528788  1.0  41.0  44.0  51.0  52.0  
High School and Below       42.623564  11.068005  1.0  40.0  44.0  52.0  52.0  
Master's and Up             45.062585   6.936764  4.0  41.0  45.0  52.0  52.0

## Multiple Groups

In [38]:
# List in groupby
nlsy_median_educSex = nlsy_working.groupby(["Education", "Sex"]).median()
nlsy_median_educSex

Income  Weeks Worked
Education                  Sex                          
Bachelor's and Associate's Female  47592.0          44.0
                           Male    70000.0          44.0
High School and Below      Female  28000.0          43.0
                           Male    42500.0          45.0
Master's and Up            Female  69000.0          45.0
                           Male    99000.0          45.0

In [39]:
# xs method
nlsy_median_educSex.xs("Female", level="Sex")

,Income,Weeks Worked
Education,,
Bachelor's and Associate's,47592.0,44.0
High School and Below,28000.0,43.0
Master's and Up,69000.0,45.0


In [41]:
# Count
nlsy_working[["Income","Sex"]].groupby("Sex").count()

,Income
Sex,
Female,2729
Male,2797


In [42]:
# Agg
nlsy_working[["Income","Sex"]].groupby("Sex").agg("count")

,Income
Sex,
Female,2729
Male,2797


In [43]:
# Multiple agg
nlsy_working[["Income","Sex"]].groupby("Sex").agg(["count", "mean", "median"])

Income                       
        count          mean   median
Sex                                 
Female   2729  48550.306706  40000.0
Male     2797  65882.961387  53000.0

In [45]:
# Multiple variables
nlsy_working[["Income", "Weeks Worked", "Sex"]].groupby("Sex").agg(["count", "mean", "median"]).loc[:,("Weeks Worked", "mean")]

Sex
Female    42.765848
Male      43.727565
Name: (Weeks Worked, mean), dtype: float64